# TrendPilot: choosing which Gen Z trends to fund, with QAOA in Qiskit
**How to run:** click **Runtime → Run all**. Wait for it to finish (a couple of minutes). Section 5 prints the table for slide 8, and Section 6 prints the table for slide 9.

**What this notebook does, in one breath:** we have 8 trends and a $10k budget. We score each trend, turn "fund it or skip it" into a yes/no variable, and let a quantum optimization algorithm (QAOA) search for the best combination. We then check it against the exact best answer.

**Everything here uses simulated data.** The trend numbers and overlaps are stand-ins for a real classical forecast (see the assumptions in Section 1).

## 0. Install Qiskit (first run only)

In [ ]:
!pip install -q qiskit qiskit-optimization scipy numpy

## 1. The problem: 8 trends, $10k
Each trend has an engagement score, a cost in $k, and a half-life in days (how fast it fades).

**The four forces from slide 6:**
- **Reward** = engagement × 0.5^(launch time ÷ half-life). Engagement that survives until launch.
- **Overlap** = penalty when two funded trends reach the same audience.
- **Waste** = spend likely to land after the trend has faded. This is the sustainability term.
- **Budget** = total cost must stay at or under $10k.

**Assumptions (say these out loud if asked):** every campaign launches 3 days after the decision. The overlap pairs and strengths below are our estimates, not measured data. A real system would learn them from audience data.

In [ ]:
import numpy as np, itertools, math

names = ["Viral dance challenge", "Y2K fashion aesthetic", "Meme format", "Sneaker drop hype",
         "Celebrity moment", "Thrift & upcycle haul", "Get-ready-with-me", "Dance sound remix"]
ENG  = np.array([90, 75, 85, 95, 80, 70, 65, 78], dtype=float)   # engagement score
COST = np.array([2, 3, 1, 4, 3, 2, 2, 2])                        # cost in $k
HALF = np.array([3, 14, 2, 6, 2, 20, 30, 3], dtype=float)        # half-life in days
N = len(names)

BUDGET = 10          # $k
LAUNCH = 3           # days from decision to launch (assumption)
LAM_WASTE = 8.0      # how strongly we penalise wasted spend (assumption)

# pairs of trends that reach the same audience, with overlap strength (assumptions)
OVERLAP = {(0, 7): 0.6,   # dance challenge  & dance sound remix
           (0, 2): 0.3,   # dance challenge  & meme format
           (2, 4): 0.3,   # meme format      & celebrity moment
           (3, 4): 0.3,   # sneaker hype     & celebrity moment
           (1, 5): 0.4,   # Y2K fashion      & thrift haul
           (5, 6): 0.2}   # thrift haul      & get-ready-with-me

def scores(eng):
    keep = 0.5 ** (LAUNCH / HALF)          # share of engagement left at launch
    reward = eng * keep
    waste  = COST * (1 - keep)             # $k spent on the part that fades before launch
    return reward, waste

def audience(x, R):
    total = sum(R[i] for i in range(N) if x[i])
    lost  = sum(f * min(R[i], R[j]) for (i, j), f in OVERLAP.items() if x[i] and x[j])
    return total - lost

def wasted(x, W):  return sum(W[i] for i in range(N) if x[i])
def spend(x):      return int(sum(COST[i] for i in range(N) if x[i]))
def feasible(x):   return spend(x) <= BUDGET
def objective(x, R, W):  return audience(x, R) - LAM_WASTE * wasted(x, W)
def chosen(x):     return [names[i] for i in range(N) if x[i]]

## 2. Two classical baselines: naive and exact

In [ ]:
def naive_top_engagement(eng):
    # fund the highest raw engagement first, until the budget runs out
    x = [0] * N; used = 0
    for i in np.argsort(-eng):
        if used + COST[i] <= BUDGET:
            x[i] = 1; used += COST[i]
    return x

def exact_best(R, W):
    # check all 2^8 = 256 portfolios
    best, best_v = None, -1e18
    for bits in itertools.product([0, 1], repeat=N):
        x = list(bits)
        if feasible(x):
            v = objective(x, R, W)
            if v > best_v: best, best_v = x, v
    return best

R, W = scores(ENG)
x_naive = naive_top_engagement(ENG)
x_exact = exact_best(R, W)
for label, x in [("Naive", x_naive), ("Exact", x_exact)]:
    print(f"{label:6s} audience {audience(x, R):6.1f}  wasted ${wasted(x, W):.1f}k  spend ${spend(x)}k  ->  {chosen(x)}")

## 3. Qiskit, step 1: turn the problem into a QUBO and a cost Hamiltonian
- `QuadraticProgram` holds the problem (yes/no variables, objective, budget).
- `QuadraticProgramToQubo` folds the budget rule into the objective, using extra "slack" bits and a penalty.
- `to_ising()` converts that into the cost Hamiltonian that QAOA works with.

In [ ]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

def build_problem(R, W):
    qp = QuadraticProgram("trendpilot")
    for i in range(N):
        qp.binary_var(name=f"x{i}")
    linear = {f"x{i}": float(R[i] - LAM_WASTE * W[i]) for i in range(N)}
    quadratic = {(f"x{i}", f"x{j}"): float(-f * min(R[i], R[j])) for (i, j), f in OVERLAP.items()}
    qp.maximize(linear=linear, quadratic=quadratic)
    qp.linear_constraint(linear={f"x{i}": int(COST[i]) for i in range(N)},
                         sense="<=", rhs=BUDGET, name="budget")
    conv = QuadraticProgramToQubo()
    qubo = conv.convert(qp)
    H, offset = qubo.to_ising()
    return conv, qubo, H

conv, qubo, H = build_problem(R, W)
print("Qubits needed:", qubo.get_num_vars(), f"({N} trend qubits + {qubo.get_num_vars() - N} budget slack qubits)")

## 4. Qiskit, step 2: QAOA
- `QAOAAnsatz` builds the circuit: start in an equal superposition of all portfolios, then alternate the cost layer and the mixer layer. **p = 2** means two rounds.
- `StatevectorEstimator` measures the circuit's energy so the classical optimizer (**COBYLA**) can tune the angles.
- `StatevectorSampler` samples the tuned circuit to get candidate portfolios. We keep the **best feasible** one (post-selection).

All of this runs on an exact, noise-free simulator.

In [ ]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize

def run_qaoa(H, conv, R, W, x_opt, reps=2, theta0=None, restarts=4, shots=8192, seed=7):
    ansatz = QAOAAnsatz(cost_operator=H, reps=reps)
    estimator = StatevectorEstimator()

    def energy(theta):
        return float(estimator.run([(ansatz, H, theta)]).result()[0].data.evs)

    rng = np.random.default_rng(seed)
    if theta0 is not None:
        starts = [np.array(theta0)]                     # warm start from yesterday's angles
    else:
        starts = [rng.uniform(0, np.pi, ansatz.num_parameters) for _ in range(restarts)]
    best = None
    for t0 in starts:
        res = minimize(energy, t0, method="COBYLA", options={"maxiter": 150})
        if best is None or res.fun < best.fun:
            best = res
    theta = best.x

    circuit = ansatz.copy()
    circuit.measure_all()
    sampler = StatevectorSampler(default_shots=shots, seed=seed)
    counts = sampler.run([(circuit, theta)]).result()[0].data.meas.get_counts()

    best_x, best_v = None, -1e18
    feas = opt = 0
    for bitstring, c in counts.items():
        bits = [int(b) for b in reversed(bitstring)]    # Qiskit prints qubit 0 on the right
        x = [int(round(v)) for v in conv.interpret(np.array(bits))][:N]
        if feasible(x):
            feas += c
            if x == list(x_opt): opt += c
            v = objective(x, R, W)
            if v > best_v: best_x, best_v = x, v
    return {"x": best_x, "theta": theta, "feasible_share": feas / shots, "optimal_share": opt / shots}

qaoa = run_qaoa(H, conv, R, W, x_exact)
print("QAOA best feasible portfolio:", chosen(qaoa["x"]))
print(f"Share of samples that were feasible: {qaoa['feasible_share']:.1%}")
print(f"Share of samples that were the exact optimum: {qaoa['optimal_share']:.1%}")

## 5. RESULTS: fills in slide 8
We also add a **random sampling** row (same number of samples, no quantum circuit). At 8 trends this is a fair sanity check, and it is why we say we do not claim quantum advantage.

In [ ]:
rng = np.random.default_rng(1)
samples = rng.integers(0, 2, size=(8192, N))
feasible_samples = [list(s) for s in samples if feasible(list(s))]
x_random = max(feasible_samples, key=lambda x: objective(x, R, W))

rows = [("Naive: top raw engagement", x_naive),
        ("Exact classical optimum", x_exact),
        ("QAOA (Qiskit, p = 2)", qaoa["x"]),
        ("Random sampling (same shots)", x_random)]
print(f"{'Strategy':32s}{'Audience':>10s}{'Wasted':>10s}{'Spend':>8s}")
for label, x in rows:
    print(f"{label:32s}{audience(x, R):10.0f}{'$' + format(wasted(x, W), '.1f') + 'k':>10s}{'$' + str(spend(x)) + 'k':>8s}")

na, ea = audience(x_naive, R), audience(x_exact, R)
nw, ew = wasted(x_naive, W), wasted(x_exact, W)
print()
print(f"Exact vs naive: audience {100 * (ea / na - 1):+.0f}%, wasted spend {100 * (ew / nw - 1):+.0f}%")
print("QAOA matches the exact optimum:", qaoa["x"] == list(x_exact))

## 6. RESULTS: fills in slide 9 (re-optimize every day)
Trends move, so we simulate five days of changing engagement (small random drift, plus a celebrity spike on day 2) and re-run each day. QAOA is **warm-started** from the previous day's angles. **This is simulated data.**

In [ ]:
rng = np.random.default_rng(3)
days = []
theta_prev = qaoa["theta"]
tot = {"naive_aud": 0, "opt_aud": 0, "naive_w": 0, "opt_w": 0}
print(f"{'Day':4s}{'QAOA = exact?':15s}{'Optimized':>10s}{'Naive':>8s}   Portfolio (QAOA)")
for d in range(5):
    eng_d = ENG * (1 + rng.normal(0, 0.06, N)) if d > 0 else ENG.copy()
    if d == 2: eng_d[4] *= 1.5                    # celebrity spike (fades in ~2 days)
    Rd, Wd = scores(eng_d)
    xe = exact_best(Rd, Wd)
    xn = naive_top_engagement(eng_d)
    conv_d, qubo_d, H_d = build_problem(Rd, Wd)
    out = run_qaoa(H_d, conv_d, Rd, Wd, xe, theta0=theta_prev)
    theta_prev = out["theta"]
    xq = out["x"]
    tot["naive_aud"] += audience(xn, Rd); tot["opt_aud"] += audience(xq, Rd)
    tot["naive_w"] += wasted(xn, Wd);     tot["opt_w"] += wasted(xq, Wd)
    print(f"{d:<4d}{str(xq == list(xe)):15s}{audience(xq, Rd):10.0f}{audience(xn, Rd):8.0f}   {chosen(xq)}")

print()
print(f"Over 5 days: audience {100 * (tot['opt_aud'] / tot['naive_aud'] - 1):+.0f}% vs naive")
print(f"Wasted spend: ${tot['naive_w']:.1f}k (naive) -> ${tot['opt_w']:.1f}k (QAOA), {100 * (tot['opt_w'] / tot['naive_w'] - 1):+.0f}%")

## What to put on the slides
Copy the numbers from the Section 5 and Section 6 printouts into slides 8 and 9. Do not reuse the old numbers from the deck, because they did not come from this notebook.

**Limits (same as slide 10):** simulator only, no quantum advantage at 8 trends, simulated data, QAOA is a heuristic, and we keep the best feasible sample.